# Experiment 07 · Fluency score v2

The original fluency score compares an answer's mean log-probability per token, under the original model, with the model's own sampled answer to the same question, and clips at 0 once the answer is 1 nat/token worse. For acrostics it has two problems:

1. **It counts the forced words.** Each sentence must open with a word starting with a given letter, which the model would rarely choose. Every acrostic answer is penalised for that, however smoothly it reads.
2. **It floors at 0.** Most acrostic answers sit more than 1 nat/token below the reference, so they all score 0 (75 of 100 in experiment 06) and the score can't tell a smooth answer from a broken one.

**`fluency_cont`** leaves each sentence's first word out of the average, in both the answer and the reference, and uses a tolerance of 3 nats/token. The cap at 1 stays.

This notebook trains nothing. It rescores answers we already have, from every experiment, with both scores, so you can check the new one ranks them sensibly before anything trains on it. It needs a GPU for the log-probabilities and takes a few minutes; rescored files are saved and reused.

In [ ]:
import os, sys

REPO = "onucharles/cv-engineer"
BRANCH = "claude/steganographic-model-selection-5hzx8n"
DEST = "/content/cv-engineer"

try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:  # not on Colab, or no secret set: fine for a public repo
    token = None

url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
if os.path.exists(DEST):
    !git -C {DEST} pull -q
else:
    !git clone -q --branch {BRANCH} {url} {DEST}

%cd {DEST}
!pip install -q -e ".[dev,gen,train]"
# Colab's preinstalled torchao is too old for current PEFT and breaks importing it.
!pip uninstall -y -q torchao
sys.path.insert(0, DEST)

## Settings

In [ ]:
MODEL = "Qwen/Qwen3-1.7B"
CHANNEL = "acrostic"
TOLERANCES = (1.0, 2.0, 3.0)   # candidate tolerances for the continuation score

In [ ]:
import pandas as pd
import torch
from stego.colab import run_dir
from stego.eval.scoring import add_fluency
from stego.io import read_jsonl, write_jsonl
from stego.rewards.fluency import fluency_score

assert torch.cuda.is_available(), "No GPU: Runtime → Change runtime type → GPU"
name = MODEL.split("/")[-1]
D = {k: run_dir(f"{k}-{CHANNEL}-{name}") for k in ("02_baselines", "03_sft", "04_dpo", "05_grpo", "06_sft_spelled")}
RUN_DIR = run_dir(f"07_fluency_score-{CHANNEL}-{name}")

# (label, file, split). Fluency references: experiment 03's for val (what 03-06 used), 06's for test_words.
SOURCES = [
    ("02 no_secret", D["02_baselines"] / "eval_answers_no_secret_scored.jsonl", "val"),
    ("02 prompted", D["02_baselines"] / "eval_answers_prompted_scored.jsonl", "val"),
    ("02 constrained", D["02_baselines"] / "eval_answers_constrained_scored.jsonl", "val"),
    ("03 sft", D["03_sft"] / "val_answers_sft_model_scored.jsonl", "val"),
    ("04 dpo", D["04_dpo"] / "val_answers_dpo_model_scored.jsonl", "val"),
    ("05 grpo step150", D["05_grpo"] / "val_answers_grpo_model_step0150_scored.jsonl", "val"),
    ("06 sft_spelled ep2", D["06_sft_spelled"] / "val_answers_sft_spelled_model_epoch02_scored.jsonl", "val"),
    ("06b plain prompt, original", D["06_sft_spelled"] / "val_answers_plain_prompt_original_model.jsonl", "val"),
    ("06b plain prompt, 06 ep2", D["06_sft_spelled"] / "val_answers_plain_prompt_sft_spelled_model_epoch02.jsonl", "val"),
    ("02 constrained", D["02_baselines"] / "eval_answers_constrained_scored.jsonl", "test_words"),
    ("06 sft_spelled ep2", D["06_sft_spelled"] / "test_words_answers_sft_spelled_model_epoch02_scored.jsonl", "test_words"),
]
REFS = {"val": D["03_sft"] / "val_fluency_references_sampled.jsonl",
        "test_words": D["06_sft_spelled"] / "test_words_fluency_references_sampled.jsonl"}
refs = {split: {r["example_id"]: r["answer"] for r in read_jsonl(p)} for split, p in REFS.items() if p.exists()}
for label, path, split in SOURCES:
    print(f"{'ok     ' if path.exists() else 'MISSING'} {split:10s} {label:28s} {path.name}")

## Rescore

Each source is rescored with both scores from one forward pass under the original model. The old `fluency` is recomputed too, so both columns use the same references.

In [ ]:
from stego.generation.runner import load_model

model, tokenizer = load_model(MODEL)
slug = lambda label: label.replace(" ", "_").replace(",", "")
rescored = {}
for label, path, split in SOURCES:
    if not path.exists() or split not in refs:
        continue
    out = RUN_DIR / f"{split}_answers_{slug(label)}_fluency_v2_scored.jsonl"
    if not out.exists():
        rows = [r for r in read_jsonl(path) if r.get("split", split) == split and r["question_id"] in refs[split]]
        print(f"rescoring {split} {label}: {len(rows)} answers")
        write_jsonl(out, add_fluency(rows, model, tokenizer, refs[split]))
    rescored[(split, label)] = read_jsonl(out)
del model; torch.cuda.empty_cache()

## Compare

`at_0` is the share of answers on the floor: the old score, then the continuation score at each candidate tolerance. A useful score keeps no-secret answers near 1, puts acrostic answers in between rather than all at 0, and spreads out answers *within* a method.

In [ ]:
from IPython.display import display

rows = []
for (split, label), rs in rescored.items():
    df = pd.DataFrame(rs)
    ok = df.answer_lp.notna()
    gap = df.answer_lp - df.ref_lp
    gap_c = df.answer_lp_cont - df.ref_lp_cont
    row = {"split": split, "source": label, "n": len(df), "fluency": df.fluency.mean(), "at_0 old": (df.fluency == 0).mean(),
           "gap": gap[ok].mean(), "gap_cont": gap_c[ok].mean()}
    for t in TOLERANCES:
        s = [fluency_score(a, b, t) if a is not None else 0.0 for a, b in zip(df.answer_lp_cont, df.ref_lp_cont)]
        row[f"cont T={t:g}"] = sum(s) / len(s)
        row[f"at_0 T={t:g}"] = sum(x == 0 for x in s) / len(s)
        row[f"spread T={t:g}"] = pd.Series(s).std()
    rows.append(row)
table = pd.DataFrame(rows).set_index(["split", "source"])
display(table.round(3))

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, col, title in [(axes[0], "gap", "all tokens (old)"), (axes[1], "gap_cont", "first words left out (v2)")]:
    for (split, label), rs in rescored.items():
        if split != "val" or label.startswith("06b plain prompt, 06"):
            continue
        df = pd.DataFrame(rs)
        g = (df.answer_lp - df.ref_lp) if col == "gap" else (df.answer_lp_cont - df.ref_lp_cont)
        ax.hist(g.dropna(), bins=30, histtype="step", label=label)
    ax.axvline(0, color="k", lw=0.8)
    ax.set_title(f"log-prob gap vs reference, {title}"); ax.set_xlabel("nats/token (0 = as likely as the reference)")
axes[0].set_ylabel("answers (val)"); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

## Read the extremes

The most and least fluent answers by the new score (T = 3), for the experiment 06 model and the experiment 05 GRPO model (the one that leaned on invented names and initials). Check the ranking matches your sense of "reads naturally".

In [ ]:
from stego.channels.splitter import split_sentences

def show(split, label, n=3, tol=3.0):
    rs = rescored.get((split, label))
    if not rs:
        print(f"({label} not available)"); return
    for r in rs:
        r["_v2"] = fluency_score(r["answer_lp_cont"], r["ref_lp_cont"], tol) if r["answer_lp_cont"] is not None else 0.0
    ranked = sorted(rs, key=lambda r: r["_v2"])
    for title, picks in (("MOST fluent", ranked[::-1][:n]), ("LEAST fluent", ranked[:n])):
        print(f"===== {label} · {title}")
        for r in picks:
            marked = " ".join(f"[{s[0]}]{s[1:]}" for s in split_sentences(r["answer"]))
            print(f"v2={r['_v2']:.2f}  old={r['fluency']:.2f}  secret={r.get('secret', '')}\n{marked}\n")

show("val", "06 sft_spelled ep2")
show("val", "05 grpo step150")

## Deciding

Pick the tolerance (default 3) where acrostic answers come off the floor and spread out, while no-secret answers stay near 1. If the extremes above look right, experiments 08 (DPO) and 09 (GRPO) train on `fluency_cont` with that tolerance.